In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
## designed to fill large Futures orders efficiently in the market
## Minimise cost
## minimise market impact
## achieve the best possible price
## Futures are standardized derivate contract that obligates the buyer to buy and seller to sell the underlying asset at a predetermined price
## at a specified date in the future
## Two key challenges: (1) buying and selling larger orders pushes price up and down massively (2) if algo takes too long the market price might
## in unfavorable direction


In [3]:
### understand more on the FuTures Algo execution methods
### learn what is Reinforcement learning
### Learn different methods of Reinforcement learning
### try and use diffrent RL library to start with and do not build the model from the start
### need a lightweight code skeleton to get started
##

In [4]:
### the agent needs to make decision how much to trade given market state


In [5]:
## learn more on the different types of reinforcement learning tommorrow and find a base model to build on
## think about where can i get the market data from
## also thinking about how to model the market impact

In [7]:
##### Start of the Code

import numpy as np
import pandas as pd

## deque is a shortcut function that helps with adding new elements to a given list while deleting the front ones, i.e old ones
from collections import deque

## helps with generating random numbers
import random


from typing import Dict, Tuple, List

## helps with creating Reinforcement learning environments
import gymnasium as gym
from gym import spaces



class FuturesExecutionEnv(gym.Env):
    """Custom Environment for Futures Execution"""
    
    ## We are setting up the room i.e. the constructor
    ## initialising order parameters
    ## if parameter values are not given, then the below values will be used
    def __init__(self, data: pd.DataFrame, order_size: int = 1000, time_horizon: int = 60, adv: float = 1000000, side: str = "BUY" ): ## int here is a type hint
        
        ## inheriting core functionality/structure from gym.Env
        ## need the below so we can properly initialize the parent class gym.Env within our FuturesExecutionEnv class
        super().__init__()
        
        assert len(data) >= time_horizon, (
        "Not enough data "
        "Each row must represent one minute, so "
        "len(data) must be >= time_horizon."
        )

        
        self.data = data  # OHLCV (Open High Low Close Volume) + order book data
        self.order_size = order_size
        self.time_horizon = time_horizon  # in minutes
        self.current_step = 0 ## setting up the time-counter to the initial historical point
        self.adv = adv
        self.side = side
        
        # Action space: we require 3 distinctive actions for now: 0=Passive, 1=Moderate, 2=Aggressive
        self.action_space = spaces.Discrete(3)
        
        # State space: we have 8 normalized features and we want to make sure these are normalized
        self.observation_space = spaces.Box(
            low = 0, high = 1, 
            shape=(8,),  # 8 state features
            dtype=np.float32
        )
        
        self.reset() ## resetting everytime
    
    
    
    ## this part is the AI agent's state or dahsboard
    ## it is compiling 8 pieces of normalized information, i.e  between 0 and 1
    def _get_state(self) -> np.ndarray: ## this function should return a numpy array (vector)
        
        """Get current state representation"""
        if self.current_step >= len(self.data) - 1: ## if the index of the current_step reaches the end, then it reverts back to the second last item
            self.current_step = len(self.data) - 2
            
        current_data = self.data.iloc[self.current_step]

        "do we need the below?"
        # next_data = self.data.iloc[self.current_step + 1]
        
        # Normalized state features
        state = np.array([
            self.remaining_quantity / self.order_size,  # % remaining
            self.current_step / self.time_horizon,         # % time elapsed
            np.tanh(current_data['spread']/0.05),      # normalized spread
            current_data['volatility'] / 0.02,         # normalized volatility
            current_data['imbalance'],                  # order book imbalance (-1 to 1)
            current_data['volume_ratio'],               # volume ratio
            self._get_urgency(),                        # execution urgency, we will define these functions in a bit
            self._get_performance()                     # current performance , we will define these funcitons in a bit 
        ], dtype=np.float32)
        
        return np.clip(state, 0, 1)                     # this limits the values in the state array between 0 and 1
    
    
    ## early on, we want to focus more on the quantity, 
    ## later on time becomes more important as we run out of trading time
    def _get_urgency(self) -> float:
        
        """Calculate execution urgency based on remaining time/quantity"""
        time_urgency = (self.current_step / len(self.data)) ## gets close to 1 during the end
        quantity_urgency = 1 - (self.remaining_quantity / self.order_size) ## close to 1 during the end
        
        if time_urgency == 1:
            return 1.0 ## Deadline reached and maximum urgency
        
        ## gets it close to 1 in the beginning and reduces to 0 as time becomes more important later on
        urgency = (
            (1 - time_urgency) * quantity_urgency +
            time_urgency * time_urgency)
        
        return float(np.clip(urgency, 0 , 1))
    
    
    
    def _get_performance(self) -> float:
        """Calculate current execution performance"""
        
        ## since we are evaluating performance between 0 and 1, we want this function to return 0.5 in the beginning since we have not executed any
        ## meaning performance is neither good nor bad
        if self.quantity_executed == 0: 
            return 0.5  # neutral
        
        ## obviously if we are long we want arrival performance to be positive
        current_vwap = self.total_value / self.quantity_executed
        arrival_performance = (self.arrival_price - current_vwap) / self.arrival_price ## price we want to buy minus price we bought at
         ## arrival_performance output will be very small, hence we will multiply it by 10 and add 0.5 since that is the start neutral point
        return np.clip(arrival_performance * 10 + 0.5, 0, 1)

    
    def _calculate_market_impact(self, action: int, quantity: int, spread: float, depth_factor: float = 0.5) -> float:
        """Realistic Market Impact model"""
   
    
    # --- 1. Passive Order → No spread, no impact ---
        if action == 0:
            return 0.0

    # Normalized volume relative to ADV (ADV stored in environment)
        vol_fraction = quantity / self.adv
    # --- 2. Transient Market Impact Component (exponential response) ---
    # η controls sensitivity:  Aggressive > Moderate > Passive
        eta = [0.0, 0.6, 1.2][action]   # you can tune these three values later
        transient_impact = eta * (vol_fraction ** 0.6)

        # --- 3. Spread Cost (only when crossing the book) ---
        # Half-spread cost for a market buy/sell
        spread_cost = spread * 0.5

        # Moderate orders cross less of the spread
        if action == 1:
            spread_cost *= 0.4      # moderate pays only ~40% of spread

        # --- 4. Depth Cost (slippage due to sweeping orderbook) ---get
        # Only for aggressive orders
        depth_cost = 0.0
        if action == 2:
            depth_cost = depth_factor * (vol_fraction ** 1.2)

        # --- Final Impact ---
        total_impact = spread_cost + transient_impact + depth_cost
        return total_impact


    def _get_execution_price(self, action: int, mid_price: float, spread: float) -> float:
        """Get execution price based on action type"""
        ## raw execution price without the market impact

        direction = 1 if self.side == "BUY" else -1

        if action == 0:      # passive
            return mid_price - direction * spread * 0.3
        elif action == 1:    # moderate
            return mid_price - direction * spread * 0.1
        else:                # aggressive
            return mid_price + direction * spread * 0.5
   
    
    def step(self, action: int):

        current_data = self.data.iloc[self.current_step]
        mid_price = current_data['close']
        spread = current_data['spread']

        base_quantity = max(1, int(self.order_size * 0.02))
        quantity_multiplier = {0: 0.5, 1: 1.0, 2: 2.0}

        quantity = min(
            int(base_quantity * quantity_multiplier[action]),
            self.remaining_quantity
        )

        if quantity > 0:
            exec_price = self._get_execution_price(action, mid_price, spread)
            impact = self._calculate_market_impact(action, quantity, spread)
            final_price = exec_price + impact

            self.quantity_executed += quantity
            self.remaining_quantity -= quantity
            self.total_value += final_price * quantity

        reward = self._calculate_reward(action, quantity)

        self.current_step += 1

        terminated = self.remaining_quantity <= 0
        truncated = self.current_step >= self.time_horizon

        return self._get_state(), reward, terminated, truncated, {}
    
    def _calculate_reward(self, action: int, quantity: int) -> float:
        """Calculate reward for the action taken"""
        
    
        if quantity == 0:
            return -0.05  # Small penalty for no execution

        current_data = self.data.iloc[self.current_step]
        spread = current_data["spread"]
        
        
        # Base reward from execution quality
        if self.quantity_executed > 0:
            current_vwap = self.total_value / self.quantity_executed

            # Universal slippage calculation (works for both buy/sell)
            if self.side == 'BUY':
                slippage = (self.arrival_price - current_vwap) / self.arrival_price
            else:  # SELL
                slippage = (current_vwap - self.arrival_price) / self.arrival_price

            # Convert to basis points and reward (positive slippage = good)
            slippage_bps = slippage * 10000
            price_reward = slippage_bps * 0.1  # Each bp = 0.1 reward
        
        else:
            price_reward = 0
        
        # Penalty for market impact
        impact_penalty = -self._calculate_market_impact(action, quantity, spread) * 100
        
        # Urgency bonus/penalty
        urgency = self._get_urgency()
        if self.remaining_quantity > 0:
            time_penalty = -0.01 if urgency > 0.8 else 0
        else:
            time_penalty = 0.1  # Bonus for completing early
        
        # Risk penalty for large remaining quantity late in execution
        risk_penalty = - (self.remaining_quantity / self.order_size) * urgency * 0.1
        
        total_reward = price_reward + impact_penalty + time_penalty + risk_penalty
        return float(total_reward)
    
    def reset(self, seed=None, options=None) -> np.ndarray:
        """Reset the environment"""
        self.current_step = 0
        self.remaining_quantity = self.order_size
        self.quantity_executed = 0
        self.total_value = 0.0
        self.arrival_price = self.data.iloc[0]['close']
        
        return self._get_state()
        

## Start of the NN logic

class DQNAgent:

    ## Q(s,a) => Expected future reward if I take action "a" in state "s"
    def __init__(self, state_size: int, action_size: int):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=2000) ## can remember the latest 2000 recent  steps
        self.gamma = 0.95  # discount rate, anything close to 1 means that the function cares more about future rewards
        
        # Start: ε = 1 → 100% random actions
        # Agent knows nothing, so explores everywhere.
        # Decay: ε *= 0.995 after each step / episode
        # Slowly prefers exploitation.
        # End: ε = 0.01 → 1% random actions
        # Almost always picks best action (according to Q-values), but still occasionally explores to avoid local optima.

        self.epsilon = 1.0  # exploration rate
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995

        self.target_update_freq  = 10
        self.step_count = 0
        self.learning_rate = 0.001 ##small number means slow, stable learning and changes mind slowly
        self.model = self._build_model() ## this is main brain function we will define next, the Q-function
        self.target_model = self._build_model() ## this is the RHS of the Bellman eqn we want to replace with as the actual function keeps on changing weights.
        self.update_target_network() ## we will update the target model (RHS of Bellman eqn) after some steps as given in the function
        
    
    def _build_model(self):
        """Build neural network model"""
        from tensorflow.keras.models import Sequential ## Used for empty model container
        from tensorflow.keras.layers import Dense
        from tensorflow.keras.optimizers import Adam

        ## this a Q-network that approximates the Q-function
        ## Predict expected future reward for each action
        ## the idea is that, we have a target Q-value from Bellman eqn and we use DQN function approximator (adj weights dynamically) to spit out predict Q-value
        ## Updates weights to minimize difference between predicted and actual Q-values
        
        model = Sequential()
        model.add(Dense(24, input_dim=self.state_size, activation='relu')) ## 1st layer of agent feeding into 24 little decision units / 1st level of feature extraction
        model.add(Dense(24, activation='relu')) ## another thinking layer for more nuanced judgements from the previous 24 outputs
        model.add(Dense(self.action_size, activation='linear')) ## takes 24 outputs from previous layer and produces self.action size / linear transformation as we need raw Q-values
        model.compile(loss='mse', optimizer=Adam(learning_rate=self.learning_rate)) ## Agent is learning by comparing predicted vs actual values
        return model
    
    def update_target_network(self):
        ## Update target network weights Q(s,a) after each transition, we want the NN to stay updated and learn
        self.target_model.set_weights(self.model.get_weights()) ## these functions are defined in TensowFlow memories
    
    def remember(self, state, action, reward, next_state, done):
        """Store experience in replay memory"""
        self.memory.append((state, action, reward, next_state, done))
    
    def act(self, state: np.ndarray) -> int:
        """Choose action using epsilon-greedy policy"""
        ## do we want to explore more or exploit
        if np.random.random() <= self.epsilon:
            return random.randrange(self.action_size) ## we will randomly exploit here without using the model
        
        state = state.reshape(1, -1) ## reshaping to array so it is consumable in the model
        act_values = self.model.predict(state, verbose=0)
        return np.argmax(act_values[0])
    
    def replay(self, batch_size: int = 32):  
        """Train on batch from replay memory"""
        if len(self.memory) < batch_size:
            return ## if memory is lesser than batch_size basically we do nothing
        
        minibatch = random.sample(self.memory, batch_size)
        
        for state, action, reward, next_state, done in minibatch:
            state = state.reshape(1, -1)
            next_state = next_state.reshape(1, -1)
            
            target = self.model.predict(state, verbose=0)
            if done: ## done is a boolean, if done = True the episode has ended
                target[0][action] = reward ## making sure we are cross matching state and target
            else:
                t = self.target_model.predict(next_state, verbose=0)
                target[0][action] = reward + self.gamma * np.amax(t[0])

            ## Adjust the neural network’s weights so that its predicted Q-values move closer to the target Q-values.
            self.model.fit(state, target, epochs=1, verbose=0)
            
        ## The agent explores a lot in the beginning, then gradually relies more on what it has learned.
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
    
    def load(self, name):
        self.model.load_weights(name)
    
    def save(self, name):
        self.model.save_weights(name)


 ## Actual futures trading algorithm           
class FuturesExecutionAlgo:
    """Main Futures Execution Algorithm"""
    
    def __init__(self, symbol: str = "ES"):
        self.symbol = symbol
        self.agent = None
        self.env = None
        self.is_trained = False
    
    def prepare_data(self, raw_data: pd.DataFrame) -> pd.DataFrame:
        """Prepare and feature engineer market data"""
        data = raw_data.copy()
        
        # Calculate features
        data['returns'] = data['close'].pct_change()
        data['volatility'] = data['returns'].rolling(20).std().fillna(0.01)
        data['spread'] = (data['ask'] - data['bid']).fillna(0.25)
       
        ## checking the bid and ask demand, if anything close to 1 then it is bid pressure
        data['imbalance'] = ((data['bid_volume'] - data['ask_volume']) / 
                           (data['bid_volume'] + data['ask_volume'])).fillna(0)
        
        ## current volume divided recent average volume
        data['volume_ratio'] = (data['volume'] / data['volume'].rolling(50).mean()).fillna(1)
        
        # Normalize features
        data['volatility'] = data['volatility'].clip(0, 0.02)
        data['spread'] = data['spread'].clip(0.1, 2.0)
        
        return data.dropna()
    
    def train(self, data: pd.DataFrame, episodes: int = 1000):
        """Train the RL agent"""
        print("Starting training...")
        
        # Prepare environment
        processed_data = self.prepare_data(data)
        self.env = FuturesExecutionEnv(processed_data)
        
        # Initialize agent
        state_size = self.env.observation_space.shape[0]
        action_size = self.env.action_space.n
        self.agent = DQNAgent(state_size, action_size)
        
        # Training metrics
        scores = []
        avg_scores = []
        
        for episode in range(episodes):
            
            ## initialising before every episode
            state = self.env.reset()
            state = state.reshape(1, -1)
            total_reward = 0
            done = False
            
            while not done: ## keep looping as long as done is False
                action = self.agent.act(state) ## agent choses an action
                next_state, reward, done, info = self.env.step(action) ## environment return next state, reward and done flag
                next_state = next_state.reshape(1, -1) ## reshape next state for NN
                
                self.agent.remember(state, action, reward, next_state, done) ## storing experience in memory
                state = next_state # move to next state
                total_reward += reward # add reward to total
            
            scores.append(total_reward) ## keeping track of total reward for this episode
            avg_score = np.mean(scores[-100:]) ## compute average reward over last 100 episodes
            avg_scores.append(avg_score) ## moving average of 100 episodes where it moves forward by 1 episode each time
            
            if episode % 100 == 0:
                print(f"Episode {episode}, Score: {total_reward:.2f}, Avg Score: {avg_score:.2f}, Epsilon: {self.agent.epsilon:.3f}")


            ## Learning from past steps: “Last time I did X in this kind of state, I got +5, but maybe doing Y next time would be even better. Let me adjust my Q-values 
            ## so I can predict the best action in similar states in the future.”
            self.agent.replay()
            
            if episode % 50 == 0:
                self.agent.update_target_network()
        
        self.is_trained = True ## obviously we want it trained before execution
        print("Training completed!")
        return scores, avg_scores
    
    def execute_order(self, live_data: pd.DataFrame, order_size: int) -> Dict:
        """Execute a live order using trained agent"""
            
        ## safety check, not allowed to trade unless the AI is trained 
        if not self.is_trained:
            raise ValueError("Agent must be trained before execution")
        
        ## data gets processed, this method was created earlier where different features were created for a dataset
        processed_data = self.prepare_data(live_data)
            
        ## Create a brand-new execution environment using this market data and this order size, and store it in self.env
        ## class is instantiated
        self.env = FuturesExecutionEnv(processed_data, order_size=order_size)
        
        state = self.env.reset() ## resetting
        done = False ## nothing executed yet
        execution_log = [] ## notebook to log actions
        
        while not done: ## keep trading until "done"
            action = self.agent.act(state) ## AI looks at features and decides what action to take
            
            ## trade gets executed, moves to next step, collates info like reward, price, how much traded, how much left
            next_state, reward, done, info = self.env.step(action)
            
            ## details get logged
            execution_log.append({
                'step': info['step'],
                'action': action,
                'quantity_executed': info['quantity_executed'],
                'remaining_quantity': info['remaining_quantity'],
                'average_price': info['avg_price'],
                'reward': reward
            })
            
            state = next_state ## update view of the market
        
        # Execution summary
        summary = {
            'total_quantity': order_size,
            'executed_quantity': info['quantity_executed'],
            'average_execution_price': info['avg_price'],
            'arrival_price': self.env.arrival_price,
            'implementation_shortfall': (self.env.arrival_price - info['avg_price']) * order_size,
            'completion_rate': info['quantity_executed'] / order_size,
            'execution_log': execution_log
        }
        
        return summary
    
    print("\nExecution Summary:")
    print(f"Executed: {result['executed_quantity']}/{result['total_quantity']}")
    print(f"Average Price: {result['average_execution_price']:.2f}")
    print(f"Arrival Price: {result['arrival_price']:.2f}")
    print(f"Implementation Shortfall: ${result['implementation_shortfall']:.2f}")

SyntaxError: unterminated triple-quoted string literal (detected at line 480) (1074827035.py, line 395)